# Studio 0: Monarch Basics - Ping Pong Tutorial

Welcome to the Lightning Studios Monarch series! This is **Studio 0**, where you'll learn the fundamentals of Monarch's Actor API through simple, hands-on examples.

> **Monarch v6 note:** This series was updated for the current Monarch API (v0.4+ / "v6"). The single-machine process-mesh API changed from `proc_mesh(gpus=N)` to `this_host().spawn_procs(per_host={"gpus": N})`. Everything in this notebook runs **locally** on your studio - no cluster needed.

## What is Monarch?

**Monarch** is Meta's distributed actor framework for building scalable, distributed applications. It makes it easy to:
- Run code across multiple processes or machines
- Coordinate distributed computations
- Build complex distributed systems with simple Python code

## What You'll Learn

1. **Core Concepts**: Actors, Endpoints, and Process Meshes
2. **Hello World**: Creating and calling actors
3. **Calling Patterns**: Broadcasting vs. targeting specific actors
4. **Actor Communication**: How actors talk to each other (Ping Pong!)

## Prerequisites

- Basic Python knowledge
- Understanding of `async`/`await`
- Monarch installed (see [installation guide](https://github.com/meta-pytorch/monarch))

## Lightning Studios Learning Path

- **Studio 0: Monarch Basics** (YOU ARE HERE)
- **[Studio 1: Getting Started](./studio_1_getting_started.ipynb)** - Multi-node training with Lightning
- **[Studio 2: Workspace Sync](./studio_2_workspace_sync.ipynb)** - Hot-reload configs without restarting
- **[Studio 3: Interactive Debugging](./studio_3_interactive_debugging.ipynb)** - Debug distributed systems

Let's dive in!

---

# Part 1: Core Concepts

## What is an Actor?

An **Actor** is an independent worker that:
- Has its own state (variables)
- Runs in its own process (possibly on a different machine)
- Exposes **endpoints** (methods) that can be called remotely

## What is an Endpoint?

An **Endpoint** is a method on an Actor that can be called remotely. It's marked with the `@endpoint` decorator.

```python
class MyActor(Actor):
    @endpoint
    async def my_method(self, arg):
        return f"Processed {arg}"
```

## What is a Process Mesh?

A **Process Mesh** (`ProcMesh`) is a collection of processes where actors can be spawned. Think of it as a cluster of workers. In v6 you create a local one with:

```python
from monarch.actor import this_host
proc_mesh = this_host().spawn_procs(per_host={"gpus": 4})   # 4 local processes
```

`this_host()` returns the current machine as a `HostMesh`; `spawn_procs` starts N worker processes on it. The dimension is named `gpus` by convention, but it works fine without GPUs - it just means "N parallel processes".

## Async/Await Quick Refresher

```python
result = await actor.my_method.call("hello")   # broadcast + wait
results = await asyncio.gather(a.m.call(), b.m.call())   # in parallel
```

---

# Part 2: Hello World

Let's create our first Monarch actor!

## Import Monarch

Note the new `this_host` import (replaces the old `proc_mesh` helper).

In [ ]:
import asyncio
from monarch.actor import Actor, current_rank, endpoint, this_host

## Define a Simple Actor

In [ ]:
NUM_ACTORS = 4


class ToyActor(Actor):
    def __init__(self):
        # Get the rank (unique ID) of this actor instance
        self.rank = current_rank().rank

    @endpoint
    async def hello_world(self, msg):
        """A simple endpoint that prints a message."""
        print(f"Actor {self.rank}: Received message '{msg}'")

### Key Points

- `Actor` base class: all Monarch actors inherit from this
- `current_rank()`: this actor's position in the mesh (`.rank` is the flat integer rank)
- `@endpoint`: makes a method remotely callable

## Create a Process Mesh and Spawn Actors

In [ ]:
async def create_toy_actors():
    # Create a LOCAL process mesh with 4 processes (v6 API).
    # Works even without actual GPUs - it just means "4 parallel processes".
    local_proc_mesh = this_host().spawn_procs(per_host={"gpus": NUM_ACTORS})

    # Spawn 4 instances of ToyActor (one per process); returns a handle to all.
    toy_actor = local_proc_mesh.spawn("toy_actor", ToyActor)

    print(f"Spawned {NUM_ACTORS} ToyActor instances")
    return toy_actor, local_proc_mesh

## Call All Actors at Once

`.call()` broadcasts to **all** instances.

In [ ]:
async def call_all_actors():
    toy_actor, local_proc_mesh = await create_toy_actors()
    await toy_actor.hello_world.call("Hello from main!")
    return toy_actor, local_proc_mesh

toy_actor, toy_mesh = await call_all_actors()

### Expected Output

```
Actor 0: Received message 'Hello from main!'
Actor 1: Received message 'Hello from main!'
Actor 2: Received message 'Hello from main!'
Actor 3: Received message 'Hello from main!'
```

---

# Part 3: Calling Specific Actors

Use `.slice()` to select specific instances.

## The Slice API

```python
actor_0 = toy_actor.slice(gpus=0)   # select the actor at index 0
await actor_0.hello_world.call_one("Hi from actor 0!")
```

## Example: Call Each Actor with a Unique Message

In [ ]:
async def call_specific_actors():
    futures = []
    for idx in range(NUM_ACTORS):
        actor_instance = toy_actor.slice(gpus=idx)
        future = actor_instance.hello_world.call_one(f"Unique message for actor {idx}")
        futures.append(future)
    # Wait for all calls to complete (in parallel!)
    await asyncio.gather(*futures)
    print("\nAll specific actor calls completed")

await call_specific_actors()

## Comparison: `.call()` vs `.call_one()`

| Method | Use Case | Example |
|--------|----------|---------|
| `.call()` | Broadcast to **all** instances | `actor.method.call(arg)` |
| `.call_one()` | Call a **specific** instance (after `.slice()`) | `actor.slice(gpus=0).method.call_one(arg)` |

---

# Part 4: Actor-to-Actor Communication (Ping Pong!)

Actors can also talk to each other.

## Define the PingPong Actor

In [ ]:
class PingPongActor(Actor):
    def __init__(self, actor_name):
        self.actor_name = actor_name
        self.identity = None
        self.other_actor = None
        self.other_actor_pair = None

    @endpoint
    async def init(self, other_actor):
        """Pair this actor with the same-rank actor in the other group."""
        self.other_actor = other_actor
        self.identity = current_rank().rank
        # Slice the other actor to get my "pair" (same coordinates).
        self.other_actor_pair = other_actor.slice(**current_rank())
        print(f"[{self.actor_name}:{self.identity}] Initialized and paired with other actor")

    @endpoint
    async def send(self, msg):
        """Send a message to our paired actor in the other group."""
        await self.other_actor_pair.recv.call(
            f"Sender ({self.actor_name}:{self.identity}) says: {msg}"
        )

    @endpoint
    async def recv(self, msg):
        """Receive a message from our paired actor."""
        print(f"Pong! [{self.actor_name}:{self.identity}] received: {msg}")

### Understanding the Code

- `init` uses `.slice(**current_rank())` to pair actors by coordinate: Actor 0 in group A pairs with Actor 0 in group B, etc.
- `send` calls `recv` on the paired actor - actor-to-actor communication!

## Create Two Actor Groups

In [ ]:
async def create_ping_pong_actors():
    local_mesh_0 = this_host().spawn_procs(per_host={"gpus": 2})
    actor_0 = local_mesh_0.spawn("actor_0", PingPongActor, "GroupA")

    local_mesh_1 = this_host().spawn_procs(per_host={"gpus": 2})
    actor_1 = local_mesh_1.spawn("actor_1", PingPongActor, "GroupB")

    print("\nCreated two actor groups (2 actors each)")
    return actor_0, actor_1, local_mesh_0, local_mesh_1

actor_group_a, actor_group_b, mesh_a, mesh_b = await create_ping_pong_actors()

## Initialize: Pair the Actors

In [ ]:
async def init_ping_pong(actor_0, actor_1):
    await asyncio.gather(
        actor_0.init.call(actor_1),  # Group A learns about Group B
        actor_1.init.call(actor_0),  # Group B learns about Group A
    )
    print("\nActors are now paired and ready to communicate!")

await init_ping_pong(actor_group_a, actor_group_b)

## Send Messages Between Actors

In [ ]:
async def send_ping_pong(actor_0, actor_1):
    print("\n" + "=" * 60)
    print("Starting Ping Pong Communication")
    print("=" * 60 + "\n")

    print("Group A sending 'Ping!' to Group B...\n")
    await actor_0.send.call("Ping!")

    print("\n" + "-" * 60 + "\n")

    print("Group B sending 'Ping!' to Group A...\n")
    await actor_1.send.call("Ping!")

    print("\n" + "=" * 60)
    print("Ping Pong Complete!")
    print("=" * 60)

await send_ping_pong(actor_group_a, actor_group_b)

### Expected Output

```
Group A sending 'Ping!' to Group B...

Pong! [GroupB:0] received: Sender (GroupA:0) says: Ping!
Pong! [GroupB:1] received: Sender (GroupA:1) says: Ping!

Group B sending 'Ping!' to Group A...

Pong! [GroupA:0] received: Sender (GroupB:0) says: Ping!
Pong! [GroupA:1] received: Sender (GroupB:1) says: Ping!
```

---

# Congratulations!

You've learned the fundamentals of Monarch:

- **Actors / Endpoints / Process Mesh**
- **`.call()`** (broadcast), **`.slice()`** + **`.call_one()`** (targeted)
- **Actor-to-actor** communication and pairing via `.slice(**current_rank())`

## Next Steps

- **[Studio 1: Getting Started](./studio_1_getting_started.ipynb)** - Multi-node training on Lightning
- **[Studio 2: Workspace Sync](./studio_2_workspace_sync.ipynb)** - Hot-reload configs
- **[Studio 3: Interactive Debugging](./studio_3_interactive_debugging.ipynb)** - Debug distributed systems

---

# Cleanup

In [ ]:
# Stop the meshes
await toy_mesh.stop()
await mesh_a.stop()
await mesh_b.stop()

print("All process meshes stopped")